# Feature engineering - advanced data preparation pipeline  | Sebislaw

## Libraries

In [1]:
from os.path  import join
import random
import itertools
import math

import numpy as np
import pandas as pd
from pandas.plotting import scatter_matrix

import matplotlib.pyplot as plt
import plotly.express as px
from pandas.plotting import parallel_coordinates
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score, StratifiedKFold, RandomizedSearchCV
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif
from sklearn.neural_network import MLPClassifier

import xgboost as xgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
# from tabpfn import TabPFNClassifier

## Data

In [2]:
data_path = '..\\..\\..\\data'
pd.set_option('display.max_columns', None)

# The Basics ------------------------------------------------------------------------
# Men
MTeams = pd.read_csv(join(data_path, 'MTeams.csv'))
MSeasons = pd.read_csv(join(data_path, 'MSeasons.csv'))
MNCAATourneySeeds = pd.read_csv(join(data_path, 'MNCAATourneySeeds.csv'))
MRegularSeasonCompactResults = pd.read_csv(join(data_path, 'MRegularSeasonCompactResults.csv'))
MNCAATourneyCompactResults = pd.read_csv(join(data_path, 'MNCAATourneyCompactResults.csv'))
# Women
WTeams = pd.read_csv(join(data_path, 'WTeams.csv'))
WSeasons = pd.read_csv(join(data_path, 'WSeasons.csv'))
WNCAATourneySeeds = pd.read_csv(join(data_path, 'WNCAATourneySeeds.csv'))
WRegularSeasonCompactResults = pd.read_csv(join(data_path, 'WRegularSeasonCompactResults.csv'))
WNCAATourneyCompactResults = pd.read_csv(join(data_path, 'WNCAATourneyCompactResults.csv'))
# Other
SampleSubmissionStage1 = pd.read_csv(join(data_path, 'SampleSubmissionStage1.csv'))
SampleSubmissionStage2 = pd.read_csv(join(data_path, 'SampleSubmissionStage2.csv'))
SeedBenchmarkStage1 = pd.read_csv(join(data_path, 'SeedBenchmarkStage1.csv'))

# Team Box Scores ------------------------------------------------------------------------
# Men
MRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'MRegularSeasonDetailedResults.csv'))
MNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'MNCAATourneyDetailedResults.csv'))
# Women
WRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'WRegularSeasonDetailedResults.csv'))
WNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'WNCAATourneyDetailedResults.csv'))

# Geography ------------------------------------------------------------------------
# All
Cities = pd.read_csv(join(data_path, 'Cities.csv'))
Conferences = pd.read_csv(join(data_path, 'Conferences.csv'))
# Men
MGameCities = pd.read_csv(join(data_path, 'MGameCities.csv'))
# Women
WGameCities = pd.read_csv(join(data_path, 'WGameCities.csv'))

# Public Rankings ------------------------------------------------------------------------
# Men
MMasseyOrdinals = pd.read_csv(join(data_path, 'MMasseyOrdinals.csv')) # men only

# Supplements ------------------------------------------------------------------------
# Men
MTeamCoaches = pd.read_csv(join(data_path, 'MTeamCoaches.csv')) # men only
MTeamConferences = pd.read_csv(join(data_path, 'MTeamConferences.csv'))
MConferenceTourneyGames = pd.read_csv(join(data_path, 'MConferenceTourneyGames.csv'))
MSecondaryTourneyTeams = pd.read_csv(join(data_path, 'MSecondaryTourneyTeams.csv'))
MSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'MSecondaryTourneyCompactResults.csv'))
MTeamSpellings = pd.read_csv(join(data_path, "MTeamSpellings.csv"), encoding='cp1252')
MNCAATourneySlots = pd.read_csv(join(data_path, 'MNCAATourneySlots.csv'))
MNCAATourneySeedRoundSlots = pd.read_csv(join(data_path, 'MNCAATourneySeedRoundSlots.csv')) # men only
# Women
WTeamConferences = pd.read_csv(join(data_path, 'WTeamConferences.csv'))
WConferenceTourneyGames = pd.read_csv(join(data_path, 'WConferenceTourneyGames.csv'))
WSecondaryTourneyTeams = pd.read_csv(join(data_path, 'WSecondaryTourneyTeams.csv'))
WSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'WSecondaryTourneyCompactResults.csv'))
WTeamSpellings = pd.read_csv(join(data_path, 'WTeamSpellings.csv'), encoding='cp1252')
WNCAATourneySlots = pd.read_csv(join(data_path, 'WNCAATourneySlots.csv'))

## Data preparation pipeline

In [3]:
def prepare_data(df):
        
    """
    This function duplicates and flips a game record.
    Now two records with the same data are present, 
    but viewed from perspectives of two different teams.
    """
    
    df = df[[
         'Season', 'DayNum', 'NumOT',
         'WTeamID',  'WScore', 'WLoc',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
         'LTeamID', 'LScore',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'
    ]]
    dfswap = df[[
         'Season', 'DayNum', 'NumOT',
         'LTeamID', 'LScore', 'WLoc',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF',
         'WTeamID',  'WScore',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
    ]].copy()
    
    dfswap.loc[df['WLoc'] == 'H', 'WLoc'] = 'A'
    dfswap.loc[df['WLoc'] == 'A', 'WLoc'] = 'H'
        
    df = df.rename(columns={'WLoc': 'location'})
    dfswap = dfswap.rename(columns={'WLoc': 'location'})
        
    df.columns = [x.replace('W','T1_').replace('L','T2_') for x in list(df.columns)]
    dfswap.columns = [x.replace('L','T1_').replace('W','T2_') for x in list(dfswap.columns)]
    
    output = pd.concat([df, dfswap]).reset_index(drop=True)
    output.loc[output.location=='N','location'] = '0'
    output.loc[output.location=='H','location'] = '1'
    output.loc[output.location=='A','location'] = '-1'
    output.location = output.location.astype(int)
        
    output['PointDiff'] = output['T1_Score'] - output['T2_Score']
    
    return output

def get_data(regular_results, tourney_results, seeds, prepared=False, location_multiplier=[1, 1], win_ratio_days_back=14):

    """
    This function uses the prepare_data function in order to create
    a data frame with season statistics for each team.
    These statistics are added to records with games played in
    tournament to make data 'x' used in model to predict the game 
    result 'y'. The output is a data frame that contains data 'x'
    and also label 'y' can be easily calculated based on score difference in matches.
    """

    if prepared:
        regular_data = regular_results.copy()
        tourney_data = tourney_results.copy()
    else:
        # make data frames with extra rows to represent the perspective of losing team
        regular_data = prepare_data(regular_results)
        tourney_data = prepare_data(tourney_results)

    # ----------------------------------- Add reward/penalty for playing in home or away
    if location_multiplier[0] == 1 and location_multiplier[1] == 1:
        # data frame with mean game statistics for a given team in a given season
        season_statistics  = regular_data.groupby(["Season", 'T1_TeamID'])[
            [
                'T1_Score', 'T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA',
                'T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF',
                'T2_Score', 'T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA',
                'T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF',
                'PointDiff'
            ]
        ].agg('mean').reset_index()
    else:
        # Define which columns to adjust (you can add or remove columns as needed)
        T1_cols = ['T1_Score','T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA','T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF']
        T2_cols = ['T2_Score','T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA','T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF']
    
        # Convert the relevant columns to float before applying the adjustment function.
        cols_to_float = T1_cols + T2_cols
        regular_data[cols_to_float] = regular_data[cols_to_float].astype(float)
        
        def adjust_stats(row):
            # Determine multipliers based on location
            if row['location'] == 1:
                factor_T1 = location_multiplier[0]  # penalize Team1 stats (home)
                factor_T2 = location_multiplier[1]  # boost Team2 stats
            elif row['location'] == -1:
                factor_T1 = location_multiplier[1]  # boost Team1 stats (away)
                factor_T2 = location_multiplier[0]  # penalize Team2 stats
            else:
                factor_T1 = 1.0
                factor_T2 = 1.0
        
            # Adjust Team1 stats
            for col in T1_cols:
                if col in row and pd.notnull(row[col]):
                    row[col] = row[col] * factor_T1
        
            # Adjust Team2 stats
            for col in T2_cols:
                if col in row and pd.notnull(row[col]):
                    row[col] = row[col] * factor_T2
        
            # Recalculate derived statistics (if needed)
            if 'T1_Score' in row and 'T2_Score' in row:
                row['PointDiff'] = row['T1_Score'] - row['T2_Score']
            return row
        
        # Apply the adjustment function row-wise.
        regular_data_adjusted = regular_data.apply(adjust_stats, axis=1)
        
        # Now group by Season and T1_TeamID to compute season averages for the adjusted statistics.
        stats_columns = T1_cols[1:] + T2_cols[1:] + ['PointDiff']  # Exclude T1_TeamID from stats if present.
        season_statistics = regular_data_adjusted.groupby(["Season", 'T1_TeamID'])[stats_columns].agg('mean').reset_index()
    # -----------------------------------
    
    # mean statistics for team and team's opponent's
    season_statistics_T1 = season_statistics.copy()
    season_statistics_T2 = season_statistics.copy()
    
    season_statistics_T1.columns = ["T1_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T1.columns)]
    season_statistics_T2.columns = ["T2_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T2.columns)]
    season_statistics_T1.columns.values[0] = "Season"
    season_statistics_T2.columns.values[0] = "Season"
    season_statistics_T1 = season_statistics_T1.rename(columns={'T1_Score': 'T1_Score_mean'})
    season_statistics_T2 = season_statistics_T2.rename(columns={'T2_Score': 'T2_Score_mean'})
    
    # data frame containing game's result
    tourney_data = tourney_data[['Season', 'DayNum', 'T1_TeamID', 'T1_Score', 'T2_TeamID' ,'T2_Score', 'location']]
    tourney_data = pd.merge(tourney_data, season_statistics_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, season_statistics_T2, on = ['Season', 'T2_TeamID'], how = 'left')

    calculate_win_ratio_days_back = 132 - win_ratio_days_back
    
    # data frame with win fraction from last x days for a given team in a given season
    last14days_stats_T1 = regular_data.loc[regular_data.DayNum>calculate_win_ratio_days_back].reset_index(drop=True)
    last14days_stats_T1['win'] = np.where(last14days_stats_T1['PointDiff']>0,1,0)
    last14days_stats_T1 = last14days_stats_T1.groupby(['Season','T1_TeamID'])['win'].mean().reset_index(name='T1_win_ratio_14d')
    
    last14days_stats_T2 = regular_data.loc[regular_data.DayNum>calculate_win_ratio_days_back].reset_index(drop=True)
    last14days_stats_T2['win'] = np.where(last14days_stats_T2['PointDiff']<0,1,0)
    last14days_stats_T2 = last14days_stats_T2.groupby(['Season','T2_TeamID'])['win'].mean().reset_index(name='T2_win_ratio_14d')
    
    # add to tourney_data column with win fraction for winning and losing team
    tourney_data = pd.merge(tourney_data, last14days_stats_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, last14days_stats_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # get seeds with no regional division
    seeds['seed'] = seeds['Seed'].apply(lambda x: int(x[1:3]))
    
    # give each team a raw seed
    seeds_T1 = seeds[['Season','TeamID','seed']].copy()
    seeds_T2 = seeds[['Season','TeamID','seed']].copy()
    seeds_T1.columns = ['Season','T1_TeamID','T1_seed']
    seeds_T2.columns = ['Season','T2_TeamID','T2_seed']
    
    # add seeds to turney data for team 1 and team 2
    tourney_data = pd.merge(tourney_data, seeds_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, seeds_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # add a seed difference column
    tourney_data["Seed_diff"] = tourney_data["T1_seed"] - tourney_data["T2_seed"]

    return tourney_data

def get_df(seeds,
            season_games, season_range, days_back,
            tourney_games, tourney_range, 
            location_multiplier=[1, 1],
           win_ratio_days_back=14):

    """
    This function uses get_data function to get a data
    frame which is then used to make 'x' and 'y' data
    used in models.
    """
    
    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]
    
    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Get final data frame
    df = get_data(regular_results, tourney_results, seeds, location_multiplier=[1, 1], win_ratio_days_back=win_ratio_days_back)
    
    return df

def get_final_df(seeds,
                   season_games, season_range, days_back,
                   tourney_games, tourney_range, 
                   SampleSubmissionStage1,
                  location_multiplier=[1, 1],
                win_ratio_days_back=14):

    """
    This function works the same as function get_x_y,
    but also creates (at the moment it's the same as get_x_y)
    aditional data points mostly with NaN values that matches
    the submission format (parsed team matchups with the sample submission file).
    """

    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]

    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Assume sample_submission is a DataFrame with an "ID" column like "2023_1101_1102"
    # and tourney_results is a DataFrame with columns including: Season, WTeamID, LTeamID, DayNum, WScore, LScore, WLoc, etc.
    # Filter rows where the ID starts with the specified season (followed by an underscore)
    final_season = tourney_range[0]
    sample_submission_copy = SampleSubmissionStage1.copy()
    sample_submission = sample_submission_copy[sample_submission_copy['ID'].str.startswith(f"{final_season}_")]
    sample_submission = sample_submission.drop(columns=['Pred'])
    
    sample_submission[['Season', 'Team1', 'Team2']] = sample_submission['ID'].str.split('_', expand=True)
    sample_submission['Season'] = sample_submission['Season'].astype(float)
    sample_submission['Team1'] = sample_submission['Team1'].astype(float)
    sample_submission['Team2'] = sample_submission['Team2'].astype(float)
    
    regular_data_final = prepare_data(regular_results)
    tourney_data_final  = prepare_data(tourney_results)
    
    tourney_data =  get_data(regular_data_final, tourney_data_final,
                             seeds, prepared=True,
                          win_ratio_days_back=win_ratio_days_back)
    tourney_data = pd.merge(
        sample_submission,
        tourney_data,
        left_on=['Season', 'Team1', 'Team2'],
        right_on=['Season', 'T1_TeamID', 'T2_TeamID'],
        how='left'
    )
    tourney_data['T1_TeamID'] = tourney_data['Team1']
    tourney_data['T2_TeamID'] = tourney_data['Team2']
    tourney_data = tourney_data.drop(['ID', 'Team1', 'Team2'], axis=1)
    
    return tourney_data

def correct_predictions_based_on_seed(x, y, maximum_favoured_seed = 4, number_of_added_columns=1):
    """
    Sets the winnning chance to 1 or 0 based on the seed difference.
    """
    for i in range(len(x)):
        if x[i][-1-number_of_added_columns] <= (-16 + maximum_favoured_seed * 2 - 1):
            y[i] = 1
        elif x[i][-1-number_of_added_columns] >= (16 - maximum_favoured_seed * 2 + 1):
            y[i] = 0
    return y

def clear_na_from_x_y(x, y):
    """
    The data frame for final season is in format matching the submission file.
    This function clears NaNs from data.
    """
    # Create masks for training data:
    mask_train = ~np.isnan(x).any(axis=1) & ~np.isnan(y)
    x_clean = x[mask_train]
    y_clean = y[mask_train]
    return x_clean, y_clean

def x_y_from_data_frame(df):
    # Prepare data and labels
    x = df[list(df.columns[7:])].values
    y = np.where(
        df[['T1_Score', 'T2_Score']].isnull().any(axis=1),
        np.nan,
        np.where(df['T1_Score'] - df['T2_Score'] > 0, 1, 0)
    )
    return x, y

def get_all_core_data(
    regular_results, tourney_results, seeds, SampleSubmissionStage1,
    final_season = 2024, # the season we want to predict, so for out submission it will be 2025
    start_season = 2005, # from which ponit should we begin creating data
    season_years_list  = [[i-1, i] for i in range(2005, 2024+1)], # at which seasons to look at when calculating team's stats
    days_back = 15, # how many days back from the start of tourney to calculate team's stats per season
    maximum_favoured_seed = 4, # Set the predicted probability of winning to 1 for seeds <= maximum_favoured_seed and to 0 for >= 16-maximum_favoured_seed
    location_multiplier=[0.95, 1.05], # home penalty, away bonus 
    include_men = True, # include M... data sets when preparing x and y
    include_women = True, # include W... data sets when preparing x and y
    win_ratio_days_back = 14):

    """
    The idea of this function is to easily get data needed to train and test the model later on, with minimal code
    to not clutter the netebook.
    
    This function outputs df, x, y, df_final, x_final_season, y_final_season.
    
    df is a data frame with first 6 columns from tourney games and other calculated from other data frames.
    
    Adding a column to df and executing x_y_from_data_frame(df) function will yield x with added data.
    
    Note that df_final has the same structure as df, but also with rows with NaNs. The rows with missing information are there
    to match the sumbission file format. The separation of those data frames is to ensure that information from last season doesn't
    leak into training data due to poorly written code.
    
    x has df columns from location onwardsthe columns before that are from tourney games and are used to calculate y (based on points).
    
    y has label 0 or 1 (lose or win) and nan if the correspoinding data in x was nan.
    
    There is also x_final_season and y_final_season aquired from df_final, which are the same as x and y, but like df_final, they have NaNs. 
    """

    # This ensures that we simulate the scenario in competition
    tourney_results_final = tourney_results[tourney_results['Season'] == final_season]
    tourney_results = tourney_results[tourney_results['Season'] < final_season]
    regular_results = regular_results[regular_results['Season'] != 2020] # This year had no tournament data
    tourney_years_list = [[i] for i in range(start_season, final_season+1)]
    
    # Arrays to store data
    data = []
    data_final = []
    for season_years, tourney_years in zip(season_years_list, tourney_years_list):
            
        # Create data separately for the of games
        # The separation is to ensure there is no data leak
        if tourney_years[0] == final_season:
            data_tmp = get_df(
                seeds,
                regular_results, season_years, days_back,
                tourney_results_final, tourney_years,
                location_multiplier=location_multiplier,
                win_ratio_days_back=win_ratio_days_back
            )
            data_final.append(data_tmp)
        else:
            data_tmp = get_df(
                seeds,
                regular_results, season_years, days_back,
                tourney_results, tourney_years,
                location_multiplier=location_multiplier,
                win_ratio_days_back=win_ratio_days_back
            )
            data.append(data_tmp)
            
    df = pd.concat(data, ignore_index=True)
    df_final = pd.concat(data_final, ignore_index=True)
    
    return df, df_final

def brier_for_all_years(year_range, season_years_list):
    
    df_train_women_list = []
    df_test_women_list = []
    df_train_men_list = []
    df_test_men_list = []
    
    for year, season_years in zip(year_range, season_years_list):
    
        final_season = year

        for sex in ['woman', 'man']:
            
            if sex == 'woman':
                include_men = False
                include_women = True
            elif sex == 'man':
                include_men = True
                include_women = False

            # ----------------------------------------------------------
            # READ DATA
            regular_results = pd.concat([
                MRegularSeasonDetailedResults.copy() if include_men else None,
                WRegularSeasonDetailedResults.copy() if include_women else None
            ], ignore_index=True)
            tourney_results = pd.concat([
                MNCAATourneyDetailedResults.copy() if include_men else None,
                WNCAATourneyDetailedResults.copy() if include_women else None
            ], ignore_index=True)
            seeds = pd.concat([
                MNCAATourneySeeds.copy() if include_men else None,
                WNCAATourneySeeds.copy() if include_women else None
            ], ignore_index=True)
            # ----------------------------------------------------------
            # GET ALL DATA NEEDED TO USE THE MODELS
            df_train, df_test = get_all_core_data(
                regular_results, tourney_results, seeds, SampleSubmissionStage1, final_season = final_season,
                start_season = start_season, season_years_list  = season_years, days_back = days_back,
                maximum_favoured_seed = maximum_favoured_seed, location_multiplier=location_multiplier,
                include_men = include_men, include_women = include_women, win_ratio_days_back = win_ratio_days_back)
            # ----------------------------------------------------------
            # ADD TEAM AND COACH ELO AND REPLACE NAN WITH MEAN
            elo = pd.read_csv(join(data_path, 'elo.csv'))
            elo['CoachELO'] = elo['CoachELO'].fillna(elo['CoachELO'].mean())
            elo = elo.drop(['CoachName'], axis=1)
            def add_elo_column(df):
                df = df.copy()
                df = pd.merge(
                        df,
                        elo[['Season', 'DayNum', 'TeamID', 'TeamELO', 'CoachELO']],
                        left_on=['Season', 'DayNum', 'T1_TeamID'],
                        right_on=['Season', 'DayNum', 'TeamID'],
                        how='left'
                    )
                df = df.drop(['TeamID'], axis=1)
                return df
            df_train = add_elo_column(df_train)
            df_test = add_elo_column(df_test)
            # ----------------------------------------------------------
            if sex == 'woman':
                df_train_women_list.append(df_train.copy())
                df_test_women_list.append(df_test.copy())
            elif sex == 'man':
                df_train_men_list.append(df_train.copy())
                df_test_men_list.append(df_test.copy())
                
        for i in range(len(df_train_men_list)):
            df_train_women_list[i] = df_train_women_list[i][list(df_train_men_list[0])]
            df_test_women_list[i] = df_test_women_list[i][list(df_train_men_list[0])]
            
    return df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list

## Get data to use models on

In [12]:
columns_to_include_women = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
 'CoachELO'
]
columns_to_include_men = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
 'CoachELO'
]

year_range = [2022]
start_season = 2010 # from which ponit should we begin creating data
season_years_list  = [[[i] for i in range(start_season, year+1)] for year in year_range] # at which seasons to look at when calculating team's stats
days_back = 25 # how many days back from the start of tourney to calculate team's stats per season
location_multiplier = [0.95, 1.05] # home penalty, away bonus ex.
win_ratio_days_back = 14 # how many days back from the tourney do we calculate win ratio
maximum_favoured_seed = 0
# -------------------------------------------
df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list = brier_for_all_years(year_range, season_years_list)
x_train_women_list = []
x_test_women_list = []
x_train_men_list = []
x_test_men_list = []
y_train_women_list = []
y_test_women_list = []
y_train_men_list = []
y_test_men_list = []
for i in range(len(year_range)):

    if len(year_range) > 1:
        df_train_women_list[i] = df_train_women_list[i][columns_to_include_women]
        df_test_women_list[i] = df_test_women_list[i][columns_to_include_women]
        df_train_men_list[i] = df_train_men_list[i][columns_to_include_men]
        df_test_men_list[i] = df_test_men_list[i][columns_to_include_men]
        
        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list[i])
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list[i])
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list[i])
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list[i])

        x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
        x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
        x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
        x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())
        
        x_train_women_list.append(x_train_women)
        x_test_women_list.append(x_test_women)
        x_train_men_list.append(x_train_men)
        x_test_men_list.append(x_test_men)

        y_train_women_list.append(y_train_women)
        y_test_women_list.append(y_test_women)
        y_train_men_list.append(y_train_men)
        y_test_men_list.append(y_test_men)
    
    else:
        df_train_women_list = df_train_women_list[0][columns_to_include_women]
        df_test_women_list = df_test_women_list[0][columns_to_include_women]
        df_train_men_list = df_train_men_list[0][columns_to_include_men]
        df_test_men_list = df_test_men_list[0][columns_to_include_men]

        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list)
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list)
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list)
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list)

        x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
        x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
        x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
        x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())

# Training models

## XGBoost

In [13]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split

############################################
# FOR WOMEN
############################################
def objective_xgb(trial, x_train, y_train, x_val, y_val):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 3, 20),
        "verbosity": 0
    }
    
    model = XGBClassifier(**params, use_label_encoder=False, eval_metric="logloss", early_stopping_rounds=50)
    model.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)
    y_pred = model.predict_proba(x_val)[:, 1]
    return brier_score_loss(y_val, y_pred)

# Split women's training data for hyperparameter tuning
x_train_women_tr, x_val_women, y_train_women_tr, y_val_women = train_test_split(
    x_train_women, y_train_women, test_size=0.2, random_state=42
)

# Run optimization for women
study_women = optuna.create_study(direction="minimize")
study_women.optimize(lambda trial: objective_xgb(trial, x_train_women_tr, y_train_women_tr, x_val_women, y_val_women), n_trials=50)
best_params_women_xgb = study_women.best_params
print("Best XGBoost Params (Women):", best_params_women_xgb)

# Train final XGBoost model for women with best hyperparameters
best_xgb_women = XGBClassifier(**best_params_women_xgb, use_label_encoder=False, eval_metric="logloss")
best_xgb_women.fit(x_train_women, y_train_women)
y_pred_women_xgb = best_xgb_women.predict_proba(x_test_women)[:, 1]
brier_women_xgb = brier_score_loss(y_test_women, y_pred_women_xgb)
print("Best XGBoost Brier Score (Women):", brier_women_xgb)

############################################
# FOR MEN
############################################
def objective_xgb_men(trial, x_train, y_train, x_val, y_val):
    # We can use the same search space as for women
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 3, 20),
        "verbosity": 0
    }
    
    model = XGBClassifier(**params, use_label_encoder=False, eval_metric="logloss", early_stopping_rounds=50)
    model.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)
    y_pred = model.predict_proba(x_val)[:, 1]
    return brier_score_loss(y_val, y_pred)

# Split men's training data for hyperparameter tuning
x_train_men_tr, x_val_men, y_train_men_tr, y_val_men = train_test_split(
    x_train_men, y_train_men, test_size=0.2, random_state=42
)

# Run optimization for men
study_men = optuna.create_study(direction="minimize")
study_men.optimize(lambda trial: objective_xgb_men(trial, x_train_men_tr, y_train_men_tr, x_val_men, y_val_men), n_trials=50)
best_params_men_xgb = study_men.best_params
print("Best XGBoost Params (Men):", best_params_men_xgb)

# Train final XGBoost model for men with best hyperparameters
best_xgb_men = XGBClassifier(**best_params_men_xgb, use_label_encoder=False, eval_metric="logloss")
best_xgb_men.fit(x_train_men, y_train_men)
y_pred_men_xgb = best_xgb_men.predict_proba(x_test_men)[:, 1]
brier_men_xgb = brier_score_loss(y_test_men, y_pred_men_xgb)
print("Best XGBoost Brier Score (Men):", brier_men_xgb)


[I 2025-03-19 22:25:07,272] A new study created in memory with name: no-name-53858aed-0c55-4003-b3ec-560253234967
[I 2025-03-19 22:25:07,992] Trial 0 finished with value: 0.13557520952842642 and parameters: {'n_estimators': 142, 'max_depth': 9, 'learning_rate': 0.018141623733080193, 'subsample': 0.6270868660873935, 'colsample_bytree': 0.9455746852885428, 'min_child_weight': 4}. Best is trial 0 with value: 0.13557520952842642.
[I 2025-03-19 22:25:09,043] Trial 1 finished with value: 0.1335664364069493 and parameters: {'n_estimators': 773, 'max_depth': 3, 'learning_rate': 0.010043008192153183, 'subsample': 0.8635966267996757, 'colsample_bytree': 0.7333046914942484, 'min_child_weight': 13}. Best is trial 1 with value: 0.1335664364069493.
[I 2025-03-19 22:25:09,466] Trial 2 finished with value: 0.13345046412352998 and parameters: {'n_estimators': 620, 'max_depth': 4, 'learning_rate': 0.04064146557856456, 'subsample': 0.7216853246235366, 'colsample_bytree': 0.7103975607117203, 'min_child_we

[I 2025-03-19 22:25:26,384] Trial 26 finished with value: 0.13005147781485057 and parameters: {'n_estimators': 594, 'max_depth': 8, 'learning_rate': 0.07026418515435799, 'subsample': 0.8030506691621835, 'colsample_bytree': 0.9166044303346438, 'min_child_weight': 8}. Best is trial 20 with value: 0.13004577940771034.
[I 2025-03-19 22:25:26,779] Trial 27 finished with value: 0.13395496766483186 and parameters: {'n_estimators': 527, 'max_depth': 7, 'learning_rate': 0.09934504811385839, 'subsample': 0.6843049978637142, 'colsample_bytree': 0.9505841501532972, 'min_child_weight': 8}. Best is trial 20 with value: 0.13004577940771034.
[I 2025-03-19 22:25:27,209] Trial 28 finished with value: 0.13260019181999008 and parameters: {'n_estimators': 573, 'max_depth': 8, 'learning_rate': 0.07421496789935847, 'subsample': 0.8010044885839902, 'colsample_bytree': 0.8702177274689429, 'min_child_weight': 7}. Best is trial 20 with value: 0.13004577940771034.
[I 2025-03-19 22:25:27,658] Trial 29 finished wit

Best XGBoost Params (Women): {'n_estimators': 630, 'max_depth': 9, 'learning_rate': 0.06863216959732682, 'subsample': 0.8230534065227971, 'colsample_bytree': 0.9601285284823977, 'min_child_weight': 13}


[I 2025-03-19 22:25:39,815] A new study created in memory with name: no-name-101d30d1-c064-4b35-be45-5810ddf61e2d


Best XGBoost Brier Score (Women): 0.21163550672863848


[I 2025-03-19 22:25:40,857] Trial 0 finished with value: 0.18145889373178592 and parameters: {'n_estimators': 515, 'max_depth': 4, 'learning_rate': 0.01325702431659338, 'subsample': 0.7773025287341081, 'colsample_bytree': 0.7532191299696638, 'min_child_weight': 5}. Best is trial 0 with value: 0.18145889373178592.
[I 2025-03-19 22:25:41,343] Trial 1 finished with value: 0.18054352311627045 and parameters: {'n_estimators': 509, 'max_depth': 3, 'learning_rate': 0.02251160000725565, 'subsample': 0.8042986401301442, 'colsample_bytree': 0.8880549039060974, 'min_child_weight': 12}. Best is trial 1 with value: 0.18054352311627045.
[I 2025-03-19 22:25:42,584] Trial 2 finished with value: 0.18157052423112938 and parameters: {'n_estimators': 911, 'max_depth': 8, 'learning_rate': 0.012100595408309335, 'subsample': 0.8355931035025217, 'colsample_bytree': 0.971402177572938, 'min_child_weight': 15}. Best is trial 1 with value: 0.18054352311627045.
[I 2025-03-19 22:25:42,887] Trial 3 finished with val

[I 2025-03-19 22:25:59,192] Trial 26 finished with value: 0.1781041563021346 and parameters: {'n_estimators': 468, 'max_depth': 9, 'learning_rate': 0.057719230491139896, 'subsample': 0.6387424122682205, 'colsample_bytree': 0.9597552048057187, 'min_child_weight': 20}. Best is trial 26 with value: 0.1781041563021346.
[I 2025-03-19 22:25:59,591] Trial 27 finished with value: 0.17942021906745562 and parameters: {'n_estimators': 580, 'max_depth': 9, 'learning_rate': 0.056998185545386236, 'subsample': 0.6416063046815127, 'colsample_bytree': 0.9394655892415552, 'min_child_weight': 20}. Best is trial 26 with value: 0.1781041563021346.
[I 2025-03-19 22:26:00,006] Trial 28 finished with value: 0.1835856544809654 and parameters: {'n_estimators': 548, 'max_depth': 9, 'learning_rate': 0.059555904978925864, 'subsample': 0.6544511723613694, 'colsample_bytree': 0.942724119503306, 'min_child_weight': 20}. Best is trial 26 with value: 0.1781041563021346.
[I 2025-03-19 22:26:00,405] Trial 29 finished wit

Best XGBoost Params (Men): {'n_estimators': 468, 'max_depth': 9, 'learning_rate': 0.057719230491139896, 'subsample': 0.6387424122682205, 'colsample_bytree': 0.9597552048057187, 'min_child_weight': 20}
Best XGBoost Brier Score (Men): 0.24650655143248526


## Logistic regression

In [16]:
def objective_lr(trial, x_train, y_train, x_val, y_val):
    C = trial.suggest_float("C", 1e-4, 10, log=True)
    
    # Define the model pipeline with scaling
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(C=C, solver="liblinear", max_iter=2000))
    ])
    
    # Train the model
    model.fit(x_train, y_train)
    
    # Predict probabilities
    y_pred = model.predict_proba(x_val)[:, 1]
    
    # Return Brier Score as the optimization metric
    return brier_score_loss(y_val, y_pred)


###################################################### WOMEN

# Split data for tuning
x_train_women_tr, x_val_women, y_train_women_tr, y_val_women = train_test_split(
    x_train_women, y_train_women, test_size=0.2, random_state=42
)

# Run Optuna optimization
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objective_lr(trial, x_train_women_tr, y_train_women_tr, x_val_women, y_val_women), n_trials=50)

# Get best parameters
best_params_women_lr = study.best_params
print("Best Logistic Regression Params (Women):", best_params_women_lr)

# Train final Logistic Regression model with best params
best_lr_women = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(C=best_params_women_lr["C"], solver="liblinear", max_iter=2000))
])
best_lr_women.fit(x_train_women, y_train_women)

# Predict on test set
y_pred_women_lr = best_lr_women.predict_proba(x_test_women)[:, 1]

# Compute Brier Score
brier_women_lr = brier_score_loss(y_test_women, y_pred_women_lr)
print("Best Logistic Regression Brier Score (Women):", brier_women_lr)


###################################################### MEN

# Split data for tuning
x_train_men_tr, x_val_men, y_train_men_tr, y_val_men = train_test_split(
    x_train_men, y_train_men, test_size=0.2, random_state=42
)

# Run Optuna optimization
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objective_lr(trial, x_train_men_tr, y_train_men_tr, x_val_men, y_val_men), n_trials=50)

# Get best parameters
best_params_men_lr = study.best_params
print("Best Logistic Regression Params (Men):", best_params_men_lr)

# Train final Logistic Regression model with best params
best_lr_men = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(C=best_params_men_lr["C"], solver="liblinear", max_iter=1000))
])
best_lr_men.fit(x_train_men, y_train_men)

# Predict on test set
y_pred_men_lr = best_lr_men.predict_proba(x_test_men)[:, 1]

# Compute Brier Score
brier_men_lr = brier_score_loss(y_test_men, y_pred_men_lr)
print("Best Logistic Regression Brier Score (Men):", brier_men_lr)

brier_men_lr2 = brier_score_loss(y_test_men, best_lr_women.predict_proba(x_test_men)[:, 1])
print("Best Logistic Regression Brier Score (Men trained on women):", brier_men_lr2)

[I 2025-03-19 22:28:13,454] A new study created in memory with name: no-name-2941386e-9eb1-4674-9304-1bf9e15119af
[I 2025-03-19 22:28:13,474] Trial 0 finished with value: 0.14051373615233612 and parameters: {'C': 9.100358441956605}. Best is trial 0 with value: 0.14051373615233612.
[I 2025-03-19 22:28:13,489] Trial 1 finished with value: 0.14042564124190965 and parameters: {'C': 0.017235187781227415}. Best is trial 1 with value: 0.14042564124190965.
[I 2025-03-19 22:28:13,492] Trial 2 finished with value: 0.19437894042073545 and parameters: {'C': 0.0005216972651030083}. Best is trial 1 with value: 0.14042564124190965.
[I 2025-03-19 22:28:13,507] Trial 3 finished with value: 0.14037416673434652 and parameters: {'C': 1.2864610266158563}. Best is trial 3 with value: 0.14037416673434652.
[I 2025-03-19 22:28:13,523] Trial 4 finished with value: 0.19364461701904953 and parameters: {'C': 0.000534953610022764}. Best is trial 3 with value: 0.14037416673434652.
[I 2025-03-19 22:28:13,529] Trial 5

[I 2025-03-19 22:28:14,306] Trial 48 finished with value: 0.13973463934537855 and parameters: {'C': 0.0956712305349025}. Best is trial 31 with value: 0.13962303768942672.
[I 2025-03-19 22:28:14,323] Trial 49 finished with value: 0.1400639417516083 and parameters: {'C': 0.2797134612898294}. Best is trial 31 with value: 0.13962303768942672.
[I 2025-03-19 22:28:14,341] A new study created in memory with name: no-name-2083aea1-be17-4b62-8b47-6cfb66cd4c8a
[I 2025-03-19 22:28:14,358] Trial 0 finished with value: 0.18950628933171978 and parameters: {'C': 4.1378578625009705}. Best is trial 0 with value: 0.18950628933171978.
[I 2025-03-19 22:28:14,374] Trial 1 finished with value: 0.1891371276522769 and parameters: {'C': 1.3630110610501838}. Best is trial 1 with value: 0.1891371276522769.
[I 2025-03-19 22:28:14,388] Trial 2 finished with value: 0.18565867994922755 and parameters: {'C': 0.026467235247730668}. Best is trial 2 with value: 0.18565867994922755.
[I 2025-03-19 22:28:14,395] Trial 3 fi

Best Logistic Regression Params (Women): {'C': 0.05160973380062567}
Best Logistic Regression Brier Score (Women): 0.16648715694525892


[I 2025-03-19 22:28:14,518] Trial 11 finished with value: 0.1862102259355363 and parameters: {'C': 0.0561820121177246}. Best is trial 2 with value: 0.18565867994922755.
[I 2025-03-19 22:28:14,537] Trial 12 finished with value: 0.1891644586900727 and parameters: {'C': 0.0045776532482132365}. Best is trial 2 with value: 0.18565867994922755.
[I 2025-03-19 22:28:14,555] Trial 13 finished with value: 0.18691780732370075 and parameters: {'C': 0.1065611347155573}. Best is trial 2 with value: 0.18565867994922755.
[I 2025-03-19 22:28:14,572] Trial 14 finished with value: 0.18563086699980036 and parameters: {'C': 0.01911451295870563}. Best is trial 14 with value: 0.18563086699980036.
[I 2025-03-19 22:28:14,585] Trial 15 finished with value: 0.23508692733140685 and parameters: {'C': 0.0001626104607935437}. Best is trial 14 with value: 0.18563086699980036.
[I 2025-03-19 22:28:14,605] Trial 16 finished with value: 0.188080686679283 and parameters: {'C': 0.31468212598388823}. Best is trial 14 with v

Best Logistic Regression Params (Men): {'C': 0.021405001207092048}
Best Logistic Regression Brier Score (Men): 0.21686416304928474
Best Logistic Regression Brier Score (Men trained on women): 0.2158504231446876


## Ensamble

In [15]:
import optuna
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier   # (Already used for CatBoost if needed)
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split, KFold
from sklearn.base import clone
from pygam import LogisticGAM, s

############################################
# (Assume you already have: x_train_women, y_train_women, x_test_women, y_test_women,
#                           x_train_men, y_train_men, x_test_men, y_test_men)
############################################

# --- Base Model Tuning for Women (already done) ---
# We assume best_lr_women and best_xgb_women are already obtained using your earlier code.
# If not, you can run your Optuna code for logistic regression and XGBoost to get them.
# For demonstration, we'll assume:
# best_lr_women: logistic regression pipeline (with StandardScaler)
# best_xgb_women: XGBoost model with optimal hyperparameters

# Similarly for men: best_lr_men and best_xgb_men are assumed available.

# For our ensemble, we want two base models per sex:
base_models_women = [best_lr_women, best_lr_women]#, best_xgb_women]
base_models_men = [best_lr_men, best_lr_women]#, best_xgb_men]

############################################
# Function to generate out-of-fold meta features using KFold CV
def generate_meta_features(model_list, X, y, n_folds=5, random_state=42):
    """
    For each model in model_list, generate out-of-fold predictions (probabilities)
    so that for each training sample we have a vector of predictions.
    """
    meta_features = np.zeros((X.shape[0], len(model_list)))
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    
    for i, model in enumerate(model_list):
        preds = np.zeros(X.shape[0])
        for train_idx, val_idx in kf.split(X):
            model_clone = clone(model)
            model_clone.fit(X[train_idx], y[train_idx])
            preds[val_idx] = model_clone.predict_proba(X[val_idx])[:, 1]
        meta_features[:, i] = preds
    return meta_features

############################################
# --- Ensemble for Women ---
# Generate meta features on training data for women
meta_train_women = generate_meta_features(base_models_women, x_train_women, y_train_women, n_folds=5)

# Train a spline meta-learner (LogisticGAM) on these meta features.
# Here we use a spline term for each base-model prediction.
gam_women = LogisticGAM(s(0) + s(1)).fit(meta_train_women, y_train_women)

# For test data, get base model predictions and combine them as meta features.
meta_test_women = np.column_stack([model.predict_proba(x_test_women)[:, 1] for model in base_models_women])
# Use the GAM meta-learner to get final probabilities.
y_pred_women_ensemble = gam_women.predict_proba(meta_test_women)
brier_women_ensemble = brier_score_loss(y_test_women, y_pred_women_ensemble)
print("Ensemble Brier Score (Women):", brier_women_ensemble)

############################################
# --- Ensemble for Men ---
meta_train_men = generate_meta_features(base_models_men, x_train_men, y_train_men, n_folds=5)
gam_men = LogisticGAM(s(0) + s(1)).fit(meta_train_men, y_train_men)

meta_test_men = np.column_stack([model.predict_proba(x_test_men)[:, 1] for model in base_models_men])
y_pred_men_ensemble = gam_men.predict_proba(meta_test_men)
brier_men_ensemble = brier_score_loss(y_test_men, y_pred_men_ensemble)
print("Ensemble Brier Score (Men):", brier_men_ensemble)

############################################
# --- Final Combined Evaluation ---
# Combine predictions and labels for final evaluation
y_pred_combined = np.concatenate([y_pred_women_ensemble, y_pred_men_ensemble])
y_test_combined = np.concatenate([y_test_women, y_test_men])
final_brier_score = brier_score_loss(y_test_combined, y_pred_combined)
print("Final Combined Brier Score:", final_brier_score)


Ensemble Brier Score (Women): 0.16674081460186166
Ensemble Brier Score (Men): 0.21788717570227553
Final Combined Brier Score: 0.19250342611910715
